[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# A Complete API &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up: the practice API, whose `serve` runs each app,
FastAPI and Pydantic. Run it first, then the tasks in order, since each one builds on the dependency
or the notes from the task before.


In [1]:
import importlib
import itertools
import sys
import urllib.request
from pathlib import Path
from typing import Annotated

import requests
from fastapi import Depends, FastAPI, HTTPException, Response, status
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from starlette.exceptions import HTTPException as StarletteHTTPException

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A dependency for a station.


In [2]:
def station_or_404(station_id: str):
    """The station with this id, or a 404 for one that does not exist."""
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app = FastAPI()


@app.get("/stations/{station_id}")
def read_station(station: Annotated[dict, Depends(station_or_404)]):
    return station


app_url = practice_api.serve(app)
for station_id in ["tromso", "bodo"]:
    response = requests.get(f"{app_url}/stations/{station_id}", timeout=10)
    print(response.status_code, response.json())


200 {'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}
404 {'detail': "no station with id 'bodo'"}


The route only returns what the dependency gave it. The `404` for Bodo was raised before the route
ran.


**2.** A created note, with its address.


In [3]:
class Note(BaseModel):
    text: str = Field(min_length=1)


notes = {}
note_ids = itertools.count(1)
app = FastAPI()


@app.post("/stations/{station_id}/notes", status_code=status.HTTP_201_CREATED)
def add_note(note: Note, response: Response, station: Annotated[dict, Depends(station_or_404)]):
    created = {"id": next(note_ids), "station": station["id"], "text": note.text}
    notes[created["id"]] = created
    response.headers["Location"] = f"/stations/{station['id']}/notes/{created['id']}"
    return created


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/stations/oslo/notes", json={"text": "the mast was checked"}, timeout=10)
print(response.status_code, response.headers["Location"], response.json())


201 /stations/oslo/notes/1 {'id': 1, 'station': 'oslo', 'text': 'the mast was checked'}


The route used the station the dependency found for the note's `station`, so a note can only belong
to a station that exists.


**3.** Two dependencies in one route.


In [4]:
def note_or_404(note_id: str):
    """The note with this id, or a 404 for one that does not exist."""
    note = notes.get(int(note_id)) if note_id.isdigit() else None
    if note is None:
        raise HTTPException(status_code=404, detail=f"no note has the id {note_id}")
    return note


app = FastAPI()


@app.get("/stations/{station_id}/notes/{note_id}")
def read_note(station: Annotated[dict, Depends(station_or_404)], note: Annotated[dict, Depends(note_or_404)]):
    return note


app_url = practice_api.serve(app)
for note_id in [1, 9]:
    response = requests.get(f"{app_url}/stations/oslo/notes/{note_id}", timeout=10)
    print(response.status_code, response.json())


200 {'id': 1, 'station': 'oslo', 'text': 'the mast was checked'}
404 {'detail': 'no note has the id 9'}


Each dependency takes its own parameter from the path, `station_id` for one and `note_id` for the
other, and FastAPI runs both before the route.


**4.** DELETE, twice.


In [5]:
app = FastAPI()


@app.delete("/stations/{station_id}/notes/{note_id}", status_code=status.HTTP_204_NO_CONTENT)
def remove_note(station: Annotated[dict, Depends(station_or_404)], note: Annotated[dict, Depends(note_or_404)]):
    del notes[note["id"]]


app_url = practice_api.serve(app)
for attempt in ["first", "second"]:
    print(attempt, requests.delete(f"{app_url}/stations/oslo/notes/1", timeout=10).status_code)


first 204
second 404


`204`, then `404`: the first request removed the note, and the second found none to remove.


**5.** Errors in one shape.


In [6]:
app = FastAPI()


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/stations/{station_id}/notes/{note_id}")
def read_note(station: Annotated[dict, Depends(station_or_404)], note: Annotated[dict, Depends(note_or_404)]):
    return note


app_url = practice_api.serve(app)
for path in ["/stations/oslo/notes/9", "/stations/oslo/note/1"]:
    print(path, requests.get(f"{app_url}{path}", timeout=10).json())


/stations/oslo/notes/9 {'error': 'no note has the id 9'}
/stations/oslo/note/1 {'error': 'Not Found'}


The dependency's `404` and FastAPI's own `404` for a path with no route came back in one shape,
because the handler is registered for Starlette's class.


**6.** The same answers as the practice API.


In [7]:
app = FastAPI()


@app.exception_handler(StarletteHTTPException)
def error_body(request, error):
    return JSONResponse(status_code=error.status_code, content={"error": error.detail}, headers=error.headers)


@app.get("/stations/{station_id}")
def read_station(station: Annotated[dict, Depends(station_or_404)]):
    return station


app_url = practice_api.serve(app)
for station_id in ["tromso", "bodo"]:
    theirs = requests.get(f"{BASE}/stations/{station_id}", timeout=10)
    ours = requests.get(f"{app_url}/stations/{station_id}", timeout=10)
    print(station_id, "| same:", theirs.status_code == ours.status_code and theirs.json() == ours.json())


tromso | same: True
bodo | same: True


Both pairs match: the station, and the `404` with its message, since `station_or_404` words its
detail as the practice API words its error.


---

&#8592; **Back to:** [A Complete API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/17-a-complete-api.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
